# Near-Field Beam Training Demo

Interactive demonstration of the CNN-based near-field beam training system.

**Paper**: J. Nie, Y. Cui et al., "Near-Field Beam Training for Extremely Large-Scale MIMO Based on Deep Learning," IEEE TMC, 2025.

In [ ]:
import sys
sys.path.insert(0, '..')

import torch
import numpy as np
import matplotlib.pyplot as plt

from src.model import BeamTrainingNet
from src.utils import (
    trans_vrf, rate_func, generate_synthetic_data,
    prepare_input_features,
)
from src.channel import NearFieldChannel
from src.beamforming import BeamformingCodebook
from src.trainer import Trainer
from src.evaluator import Evaluator

print('All modules imported successfully!')

## 1. Explore the Near-Field Channel Model

In [ ]:
# Create near-field channel model
channel = NearFieldChannel(num_antennas=256, wavelength=0.01)

# Generate a single channel at 50m, broadside
h = channel.generate_channel(distance=50.0, angle=0.0)
print(f'Channel shape: {h.shape}')
print(f'Average power: {np.mean(np.abs(h)**2):.6f}')

# Visualize phase profile across array
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(12, 4))
ax1.plot(np.angle(h))
ax1.set_xlabel('Antenna Index')
ax1.set_ylabel('Phase (rad)')
ax1.set_title('Near-Field Channel Phase Profile')
ax1.grid(True, alpha=0.3)

ax2.plot(np.abs(h))
ax2.set_xlabel('Antenna Index')
ax2.set_ylabel('Magnitude')
ax2.set_title('Near-Field Channel Magnitude')
ax2.grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

## 2. Generate Synthetic Data & Train

In [ ]:
# Quick training demo (few epochs)
config = {
    'num_antennas': 256,
    'batch_size': 64,
    'num_epochs': 20,
    'learning_rate': 0.001,
    'lr_factor': 0.5,
    'lr_patience': 10,
    'min_lr': 1e-4,
    'val_split': 0.2,
    'num_synthetic_samples': 2000,
    'seed': 42,
    'log_interval': 5,
    'checkpoint_dir': '/tmp/demo_checkpoints',
}

trainer = Trainer(config, device='cpu')
trainer.setup_model()
trainer.load_data()
history = trainer.train()

# Plot training curves
plt.figure(figsize=(8, 4))
plt.plot(history['train_loss'], label='Train')
plt.plot(history['val_loss'], label='Validation')
plt.xlabel('Epoch')
plt.ylabel('Loss (-Rate)')
plt.title('Training Progress')
plt.legend()
plt.grid(True, alpha=0.3)
plt.show()

## 3. Evaluate Performance

In [ ]:
# Evaluate on test data
H_test, H_est_test = generate_synthetic_data(num_samples=500, seed=123)

evaluator = Evaluator(trainer.model, device='cpu')
metrics = evaluator.evaluate_all_metrics(H_test, H_est_test)

# Plot Rate vs SNR
evaluator.plot_rate_vs_snr(
    metrics['snr_dB'],
    metrics['spectral_efficiency'],
    title='Demo: Rate vs SNR',
)

print(f"Avg Beamforming Gain: {metrics['avg_beamforming_gain_dB']:.2f} dB")
print(f"Normalized MSE: {metrics['normalized_mse']:.6f}")

## 4. Visualize Beamforming Vectors

In [ ]:
# Compare CNN-predicted vs optimal (MRT) beamforming
from src.utils import prepare_input_features

model = trainer.model
model.eval()

sample_idx = 0
h_sample = H_test[sample_idx]
h_est_sample = H_est_test[sample_idx:sample_idx+1]

# CNN prediction
h_input = prepare_input_features(h_est_sample)
h_input_t = torch.tensor(h_input, dtype=torch.float32)
with torch.no_grad():
    phases = model(h_input_t)
    v_cnn = trans_vrf(phases).numpy()[0]

# Optimal MRT
v_mrt = h_sample / np.linalg.norm(h_sample)

# Plot phase comparison
fig, axes = plt.subplots(2, 2, figsize=(12, 8))

axes[0, 0].plot(np.angle(h_sample))
axes[0, 0].set_title('True Channel Phase')
axes[0, 0].set_ylabel('Phase (rad)')

axes[0, 1].plot(np.angle(h_est_sample[0]))
axes[0, 1].set_title('Estimated Channel Phase')

axes[1, 0].plot(np.angle(v_cnn))
axes[1, 0].set_title('CNN-Predicted Beamformer Phase')
axes[1, 0].set_xlabel('Antenna Index')
axes[1, 0].set_ylabel('Phase (rad)')

axes[1, 1].plot(np.angle(v_mrt))
axes[1, 1].set_title('Optimal (MRT) Beamformer Phase')
axes[1, 1].set_xlabel('Antenna Index')

for ax in axes.flat:
    ax.grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

# Compare beamforming gains
gain_cnn = np.abs(np.vdot(h_sample, v_cnn))**2
gain_mrt = np.abs(np.vdot(h_sample, v_mrt))**2
print(f'CNN beamforming gain: {gain_cnn:.4f}')
print(f'MRT beamforming gain: {gain_mrt:.4f}')
print(f'Efficiency: {gain_cnn/gain_mrt*100:.1f}%')